# Synapse vs Databricks Data Validation

Reusable notebook that compares tables between Synapse and Databricks.
Define your table mappings below, then run all cells to get a full comparison report.

**Requirements:** `pyodbc`, `polars`, `databricks-sql-connector`

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, os.path.abspath('..'))

import pyodbc
from databricks import sql
import polars as pl

from validation_utils import compare_synapse_vs_databricks

## Configuration

Edit the table mappings and connection details below.

In [ ]:
# ─── Table Mappings ─────────────────────────────────────────────────────────────
# Dict of source_system → list of (synapse_schema, synapse_table, dbx_catalog, dbx_schema, dbx_table)

TABLE_MAPPINGS = {
    "IOC_MMRS": [
        ("hstg_v", "vw_hstg_UDLMMRSIOC_mmrs_cs_vw_cycle", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_cycle"),
        ("hstg_v", "vw_hstg_UDLMMRSIOC_mmrs_cs_vw_event", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_event"),
        ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_location", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_location"),
        ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_rf_material", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_material"),
        ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_rf_timecode", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_timecode"),
        ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_rf_unit", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_unit"),
        ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_unit_fleet_fleettype", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_unit_fleet_fleettype"),
    ],
    "GENSD03_SICOM": [
        ("HSTG_V", "VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_suivietatequipementrownumber_view"),
        ("HSTG_V", "VW_HSTG_UDLGENSD03_DBO_GEN_CODEIPT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_gen_codeipt_def"),
        ("HSTG_V", "VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_suivietatequipement"),
        ("HSTG_V", "VW_HSTG_UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_groupeequipement_def"),
        ("HSTG_V", "VW_HSTG_UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_modeleequipement_def"),
        ("HSTG_V", "VW_HSTG_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_equipement_def"),
    ],
    "PI_SYSTEMS": [
        ("hstg_v", "vw_hstg_udlpisystems_opsdata_mis_rtit_headgrade", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_mis_rtit_headgrade"),
        ("hstg_v", "vw_hstg_udlpisystems_opsdata_cs_mis_rtit_sandmetrics", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_mis_rtit_sandmetrics"),
        ("hstg_v", "vw_hstg_udlpisystems_opsdata_cs_pm_reporting_data", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_pm_reporting_data"),
        ("hstg_v", "vw_hstg_udlpisystems_opsdata_cs_pm_reporting_recovery_data", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_pm_reporting_recovery_data"),
        ("hstg_v", "vw_hstg_udlpisystems_pm_reporting_locations_hierarchy", "lakehouse", "analytics_and_enablement_staging_pi_systems", "globalpi_pm_reporting_locations_hierarchy"),
    ],
    "ODP": [
# dbx        ("hstg_v", "vw_hstg_UDLDWBDP_DAL_CS_FACT_HMETIMEUSAGEEVENT", "ogr_sourcealigned", "odp", "dal_fact_hmetimeusageevent"),
# dbx       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_HMEASSET", "ogr_sourcealigned", "odp", "dal_dim_hmeasset"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_HMETIMEUSAGEMODEL", "ogr_sourcealigned", "odp", "dal_dim_hmetimeusagemodel"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_LOCATION", "ogr_sourcealigned", "odp", "dal_dim_location"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_PAYLOADMANAGEMENT", "ogr_sourcealigned", "odp", "dal_dim_payloadmanagement"),
# dbx       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_FACT_HMELOADHAULCYCLE", "ogr_sourcealigned", "odp", "dal_fact_hmeloadhaulcycle"),
        ("hstg_v", "vw_hstg_UDLODP_DAL_DIM_FIXEDPLANTMETRIC", "ogr_sourcealigned", "odp", "dal_dim_fixedplantmetric"),
        ("hstg_v", "vw_hstg_UDLODP_DAL_DIM_MATERIAL", "ogr_sourcealigned", "odp", "dal_dim_material"),
        ("hstg_v", "vw_hstg_UDLODP_DAL_DIM_SHIFT", "ogr_sourcealigned", "odp", "dal_dim_shift"),
        ("hstg_v", "vw_hstg_UDLODP_DAL_DIM_SITE", "ogr_sourcealigned", "odp", "dal_dim_site"),
        ("hstg_v", "vw_hstg_UDLODP_DAL_DIM_TARGET", "ogr_sourcealigned", "odp", "dal_dim_target"),
        ("hstg_v", "vw_hstg_UDLODP_DAL_FACT_FIXEDPLANTPRODUCTIONSHIFT", "ogr_sourcealigned", "odp", "dal_fact_fixedplantproductionshift"),
        ("hstg_v", "vw_hstg_UDLODP_DAL_FACT_FIXEDPLANTTARGET", "ogr_sourcealigned", "odp", "dal_fact_fixedplanttarget"),
    ],
        "EGRP_GREPORTS": [
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G42413A_ZBB_QUALITY_LOCAL_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g42413a_zbb_quality_local_gi_governed"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44561A_SCHEDULE_COMPLIANCE_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44561a_schedule_compliance_gi_governed"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44561A_SCHEDULE_WORK_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44561a_schedule_work_gi_governed"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44564A_SNAPSHOT_BACKLOG_AGE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44564a_snapshot_backlog_age_governed"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44565A_USE_OF_RESOURCE_AVAILABILITY_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44565a_use_of_resource_availability_gi_governed"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44567A_PRIMARY_MAINTENANCE_CALL_COMPLIANCE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44567a_primary_maintenance_call_compliance_governed"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44569A_MAINTENANCE_PLANS_FUNCTIONAL_LOCATION_CRI_LOOKUP", "ent_sourcealigned", "egrp_greports", "g44569a_maintenance_plans_functional_location_cri_lookup"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44576A_WORK_EVENT_PROFILE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44576a_work_event_profile_governed"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G52002A_PRODUCT_QUALITY", "ent_sourcealigned", "egrp_greports", "g52002a_product_quality"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G52004A_TOTAL_MATERIAL_MOVED_ACTUAL", "ent_sourcealigned", "egrp_greports", "g52004a_total_material_moved_actual"),
        ("hstg_v", "vw_hstg_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G70301A_MAT_MVMT_ANLYS_PRODFACTSHEET_GOV", "ent_sourcealigned", "egrp_greports", "g70301a_mat_mvmt_anlys_prodfactsheet_gov"),
    ],
    "EGRP_IAP_STAGING": [
# syp        ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MMP_RDL_DIM_PROCESS_HIER", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mmp_rdl_dim_process_hier"),
        ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MP_RDL_DIM_OPERATING_RESPONSIBILITY", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mp_rdl_dim_operating_responsibility"),
# syp       ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MP_RDL_DIM_OPERATING_RESPONSIBILITY_HIER", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mp_rdl_dim_operating_responsibility_hier"),
        ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MPA_ODL_DIM_FUNCTIONAL_LOCATION", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mpa_odl_dim_functional_location"),
# syp       ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MPA_RDL_DIM_ASSET", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mpa_rdl_dim_asset"),
    ],
    "EGRP_SAP_BRONZE": [
        ("hstg_v", "vw_hstg_UDLLIBECPRTP_SAP_CS_QMFE", "ent_sourcealigned", "egrp_sap_bronze", "qmfe"),
        ("hstg_v", "vw_hstg_UDLLIBECPRTP_SAP_MAKT", "lakehouse", "lib_ecp_rtp", "sap_makt"),
    ],
}

In [ ]:
# ─── Connection Setup ──────────────────────────────────────────────────────────

# Synapse
synapse_server = 'syn-rt-prd-unity.sql.azuresynapse.net'
synapse_database = 'syndwrtprdunity'
synapse_username = 'byambadorjme@riotinto.com'
synapse_driver = '{ODBC Driver 17 for SQL Server}'

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""

# Databricks
databricks_access_token = os.environ["DATABRICKS_TOKEN"]
databricks_server_hostname = 'adb-2439498039309203.3.azuredatabricks.net'
databricks_http_path = '/sql/1.0/warehouses/e7d134c4348ac8b5'

# ─── Connect ──────────────────────────────────────────────────────────────────
print("Connecting to Synapse (browser auth prompt may appear)...")
synapse_conn = pyodbc.connect(synapse_connection_string)
print("✓ Synapse connected")

databricks_conn = sql.connect(
    server_hostname=databricks_server_hostname,
    http_path=databricks_http_path,
    access_token=databricks_access_token,
)
print("✓ Databricks connected")

In [ ]:
# ─── Display settings ─────────────────────────────────────────────────────────
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(100)

## Run Comparison

Iterates over all table mappings and produces a summary report.

In [ ]:
# ─── Run all comparisons ──────────────────────────────────────────────────────
all_results = {}

#for source_system, mappings in TABLE_MAPPINGS.items():

source_system = 'EGRP_SAP_BRONZE'

print(f"\n{'█' * 80}")
mappings = TABLE_MAPPINGS[source_system]
print(f"  SOURCE SYSTEM: {source_system} ({len(mappings)} tables)")
print(f"{'█' * 80}")

system_results = []

for i, (syn_schema, syn_table, dbx_catalog, dbx_schema, dbx_table) in enumerate(mappings, 1):
    print(f"\n{'═' * 80}")
    print(f"  [{i}/{len(mappings)}] {syn_schema}.{syn_table} → {dbx_catalog}.{dbx_schema}.{dbx_table}")
    print(f"{'═' * 80}")

    try:
        result = compare_synapse_vs_databricks(
            databricks_conn=databricks_conn,
            synapse_conn=synapse_conn,
            synapse_schema_name=syn_schema,
            synapse_table_name=syn_table,
            databricks_catalog_name=dbx_catalog,
            databricks_schema_name=dbx_schema,
            databricks_table_name=dbx_table,
        )
        result["source_system"] = source_system
        result["synapse_table"] = syn_table
        result["databricks_table"] = dbx_table
        system_results.append(result)

        print(f"  Schema Match: {result['schema_matches']}")
        print(f"  Row Count Match: {result['row_count_matches']}")
        print(f"  Synapse: {result['synapse_row_count']:,} | Databricks: {result['databricks_row_count']:,}")
        if result['sampling_applied']:
            print(f"  ** Sampled (last 30 days): Syn={result['sampled_row_count_synapse']:,} | Dbx={result['sampled_row_count_databricks']:,}")
        print(f"  Data Match: {result['matched_rows']:,}/{result['syn_total']:,} ({result['pct_of_synapse']:.2f}%)")
        print(f"  Hash Match: {result['full_match']}")

    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        system_results.append({
            "source_system": source_system,
            "synapse_table": syn_table,
            "databricks_table": dbx_table,
            "error": str(e),
        })

    all_results[source_system] = system_results

total_tables = sum(len(v) for v in all_results.values())
print(f"\n{'█' * 80}")
print(f"  DONE: {total_tables} tables compared across {len(all_results)} source systems")
print(f"{'█' * 80}")

In [ ]:
# ─── Run all comparisons ──────────────────────────────────────────────────────
all_results = {}

for source_system, mappings in EGRP_MAPPINGS.items():
    print(f"\n{'█' * 80}")
    print(f"  SOURCE SYSTEM: {source_system} ({len(mappings)} tables)")
    print(f"{'█' * 80}")

    system_results = []

    for i, (syn_schema, syn_table, dbx_catalog, dbx_schema, dbx_table) in enumerate(mappings, 1):
        print(f"\n{'═' * 80}")
        print(f"  [{i}/{len(mappings)}] {syn_schema}.{syn_table} → {dbx_catalog}.{dbx_schema}.{dbx_table}")
        print(f"{'═' * 80}")

        try:
            result = compare_synapse_vs_databricks(
                databricks_conn=databricks_conn,
                synapse_conn=synapse_conn,
                synapse_schema_name=syn_schema,
                synapse_table_name=syn_table,
                databricks_catalog_name=dbx_catalog,
                databricks_schema_name=dbx_schema,
                databricks_table_name=dbx_table,
            )
            result["source_system"] = source_system
            result["synapse_table"] = syn_table
            result["databricks_table"] = dbx_table
            system_results.append(result)

            print(f"  Schema Match: {result['schema_matches']}")
            print(f"  Row Count Match: {result['row_count_matches']}")
            print(f"  Synapse: {result['synapse_row_count']:,} | Databricks: {result['databricks_row_count']:,}")
            if result['sampling_applied']:
                print(f"  ** Sampled (last 30 days): Syn={result['sampled_row_count_synapse']:,} | Dbx={result['sampled_row_count_databricks']:,}")
            print(f"  Data Match: {result['matched_rows']:,}/{result['syn_total']:,} ({result['pct_of_synapse']:.2f}%)")
            print(f"  Hash Match: {result['full_match']}")

        except Exception as e:
            print(f"  ✗ ERROR: {e}")
            system_results.append({
                "source_system": source_system,
                "synapse_table": syn_table,
                "databricks_table": dbx_table,
                "error": str(e),
            })

    all_results[source_system] = system_results

total_tables = sum(len(v) for v in all_results.values())
print(f"\n{'█' * 80}")
print(f"  DONE: {total_tables} tables compared across {len(all_results)} source systems")
print(f"{'█' * 80}")

In [ ]:
# ─── Summary Report ───────────────────────────────────────────────────────────
summary_rows = []
for source_system, results in all_results.items():
    for r in results:
        if "error" in r:
            summary_rows.append({
                "source_system": source_system,
                "table": r["databricks_table"],
                "synapse_rows": None,
                "databricks_rows": None,
                "schema_match": None,
                "row_count_match": None,
                "data_match_pct": None,
                "hash_match": None,
                "status": f"ERROR: {r['error'][:50]}",
            })
        else:
            status = "✓ Perfect" if r["full_match"] else (
                "⚠ Row count mismatch" if not r["row_count_matches"] else "⚠ Data mismatch"
            )
            summary_rows.append({
                "source_system": source_system,
                "table": r["databricks_table"],
                "synapse_rows": r["synapse_row_count"],
                "databricks_rows": r["databricks_row_count"],
                "schema_match": r["schema_matches"],
                "row_count_match": r["row_count_matches"],
                "data_match_pct": round(r["pct_of_synapse"], 2),
                "hash_match": r["full_match"],
                "status": status,
            })

summary_df = pl.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
print(summary_df)

In [ ]:
# ─── Detailed Results (per table) ─────────────────────────────────────────────
# Uncomment a table index to inspect its detailed results

# idx = 0  # Change index to inspect a specific table
# r = all_results['GENSD03_SICOM'][idx]
# print(f"Table: {r['databricks_table']}")
# print(f"\nSchema Comparison:")
# display(r['schema_comparison_df'])
# print(f"\nNumeric Profile:")
# display(r['numeric_profile_comparison_df'])
# print(f"\nString Profile:")
# display(r['string_profile_comparison_df'])

In [ ]:
# ─── Cleanup ──────────────────────────────────────────────────────────────────
synapse_conn.close()
databricks_conn.close()
print("Connections closed.")